# Quantum Oracle Sketching — PCA Dimension Reduction

> Fourth notebook in the QOS series. Where the [LS-SVM notebook](qos_lssvm.ipynb) used the QSVT *inversion* polynomial to solve a linear system, this one uses the QSVT *spectral-projector* polynomial to extract the top principal direction of a feature matrix — the building block of [principal-component analysis (PCA)](https://en.wikipedia.org/wiki/Principal_component_analysis). Reproduces the spirit of Theorem 5 (Sec. F.3) of [[1]](#ref1).
>
> The pipeline is a near-twin of the LS-SVM one — only the QSVT polynomial changes:
>
> - **Stage 1.** Sketch the matrix-element phase oracle of the centred Gram matrix $X^\top X$ (or kernel matrix) via `oracle_sketching.matrix_element_sketch`.
> - **Stage 2.** Apply the QSVT *projector* polynomial $\Pi_\text{top}(x) \approx \tfrac{1}{2}(1 + \mathrm{sgn}(x - \mu))$ from `quantum_ml.qsvt_projector_polynomial`, with $\mu$ chosen between the top and second eigenvalues of the normalised Gram matrix. The result is a block encoding of the spectral projector onto the top-eigenvalue subspace.
> - **Stage 3.** Apply the projector to a *guiding vector* $\vec g$ — a classical estimate of the top principal direction, with $\langle \vec g | \vec w \rangle$ bounded away from zero — to recover the top eigenvector $\vec w$. Test points are then reduced to a single scalar $\xi(\vec x_\text{test}) = \vec w \cdot \vec x_\text{test}$, again via interferometric-shadow readout (`classical_shadow.shadow_readout_proxy` for the proxy used here).
>
> ---
>
> **Inputs.** $M$ classical samples of the matrix entries of $X^\top X$; a guiding vector $\vec g \in \mathbb{R}^D$ with $|\vec g \cdot \vec w| \ge \Omega(1)$.
>
> **Output.** Top principal direction $\vec w_\text{quantum} \in \mathbb{R}^D$ with $|\vec w_\text{quantum} \cdot \vec w_\text{classical}| \to 1$ as $M \to \infty$, and an explained-variance ratio that matches the classical PCA baseline.
>
> **Sample complexity.** $M = \widetilde{\mathcal{O}}(D^2 / \Delta^2)$ for a spectral gap $\Delta$ between the top and second eigenvalues — exponentially less than the $\Omega(D^{0.99})$ memory floor of any classical machine on dynamic data (Theorem 6 of [[1]](#ref1)).
>
> ---
>
> **Keywords:** Quantum Oracle Sketching, QSVT, PCA, Spectral Projector, Quantum Machine Learning, Dimension Reduction

## PCA in one line

Given a centred feature matrix $X \in \mathbb{R}^{N\times D}$ (rows are samples, columns are features), the top *principal component* is the unit vector $\vec w \in \mathbb{R}^D$ that maximises the sample variance:

$$
\vec w \;=\; \arg\max_{\|\vec u\| = 1} \; \vec u^\top X^\top X \, \vec u.
$$

The maximiser is the top eigenvector of the **Gram / scatter matrix** $A = X^\top X$, which can be extracted as the limit of $A^k \vec g$ for any starting vector $\vec g$ with non-zero overlap on $\vec w$ (the power-iteration intuition).

The QSVT alternative replaces the power iteration by a *single* spectral-projector application: pick a threshold $\mu \in (\lambda_2, \lambda_1)$ between the second and top eigenvalues of $A$, and apply the polynomial

$$
\Pi_\text{top}(x) \;=\; \tfrac{1}{2}\,\bigl(1 + \mathrm{sgn}(x - \mu)\bigr).
$$

On the eigenbasis of $A$, this projector zeroes out every eigenmode below $\mu$ and preserves every eigenmode above — a single block encoding gives a clean projection. Smoothing $\mathrm{sgn}$ over a transition region of width $\delta$ keeps the polynomial degree bounded.

For the toy demonstration below we use $D = 8$ features, a planted rank-1 signal in a known direction, and Gaussian noise small enough to leave the spectral gap $\lambda_1 - \lambda_2$ intact.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from oracle_sketching import (
    apply_basis_phase_2d,
    matrix_element_oracle_sketch,
)
from quantum_ml import quantum_pca_top_eigenvector

from classiq import *

np.random.seed(7)

### Toy data

Plant a rank-1 signal in a known direction $\vec w_\text{true}$ and add small Gaussian noise. The signal-to-noise ratio is tuned so the top eigenvalue is well-separated from the second (a spectral gap of order $\Delta \sim 0.5$ in normalised units).

In [ ]:
D = 8  # feature dimension
N_SAMPLES = 64  # number of data rows
SIGNAL = 1.0  # variance contributed by the planted direction
NOISE = 0.15  # isotropic noise std

# Planted top principal direction.
w_true = np.random.randn(D)
w_true /= np.linalg.norm(w_true)

# Generate data matrix: each row is a · w_true + noise, with a ~ N(0, SIGNAL).
amplitudes = SIGNAL * np.random.randn(N_SAMPLES)
X = amplitudes[:, None] * w_true[None, :] + NOISE * np.random.randn(N_SAMPLES, D)

# Normalise so |X_{ij}| ≤ 1 (matrix-element sketch domain).
X /= np.max(np.abs(X))

# Centre the data (standard PCA preprocessing).
X = X - X.mean(axis=0, keepdims=True)

print(f"feature dim D       = {D}")
print(f"number of rows N    = {N_SAMPLES}")
print(f"|X|_max after norm  = {np.max(np.abs(X)):.3f}")

### Classical PCA baseline

In [ ]:
A_target = X.T @ X
eigvals, eigvecs = np.linalg.eigh(A_target)

# eigh returns eigenvalues in ascending order; the top one is the last column.
w_classical = eigvecs[:, -1]
lambda_top, lambda_second = float(eigvals[-1]), float(eigvals[-2])
spectral_gap = lambda_top - lambda_second

print(f"top eigenvalue λ₁     = {lambda_top:.3f}")
print(f"second eigenvalue λ₂  = {lambda_second:.3f}")
print(f"spectral gap Δ        = {spectral_gap:.3f}")
print(f"|⟨w_classical | w_true⟩| = {abs(w_classical @ w_true):.4f}    (1 = perfect)")
print()
print(f"Explained variance ratio λ₁ / Σ λ = {lambda_top / np.sum(eigvals):.3f}")

## Quantum pipeline

We sketch $A = X^\top X$ via the matrix-element phase oracle, normalise it to $A_\text{norm} = A / \alpha$ with $\alpha = \|A\|_2$ so its eigenvalues sit in $[0, 1]$, and then apply the QSVT projector polynomial centred at $\mu_\text{norm} = (\lambda_1 + \lambda_2) / (2\alpha)$ — the midpoint of the spectral gap. The polynomial is built by `quantum_ml.chebyshev_projector_polynomial` and applied via `quantum_ml.apply_qsvt_polynomial`. The recovered top eigenvector is then extracted by projecting a guiding vector $\vec g$ (here we use a noisy version of $\vec w_\text{true}$, mimicking the "good initial estimate" assumed by Theorem 5).

In [ ]:
# Guiding vector: noisy version of w_true, mimicking the "good initial estimate"
# assumption of Theorem 5 of [[1]]. The bound |⟨g|w⟩| ≥ Ω(1) keeps the
# post-projector renormalisation factor non-negligible.
g = w_true + 0.5 * np.random.randn(D)
g /= np.linalg.norm(g)
print(f"|⟨g | w_classical⟩| = {abs(g @ w_classical):.3f}    (must be ≥ Ω(1))")

# Use the imported `quantum_pca_top_eigenvector` from `quantum_ml.py`.
w_quantum, pca_diag = quantum_pca_top_eigenvector(
    X,
    g,
    spectral_gap_lower=lambda_second / lambda_top,  # using α ≈ λ_top so normalised
    spectral_gap_upper=1.0,
    rng=np.random.default_rng(11),
)

print(f"\nA recovery error  = {pca_diag['A_recovery_error']:.3e}")
print(f"α (spectral norm) = {pca_diag['alpha']:.3f}")
print(f"μ_norm            = {pca_diag['mu_norm']:.3f}")
print()
overlap = abs(w_quantum @ w_classical)
print(f"|⟨w_quantum | w_classical⟩| = {overlap:.4f}    (1 = perfect)")
print(f"|⟨w_quantum | w_true⟩|      = {abs(w_quantum @ w_true):.4f}")

### Explained-variance verification

Project the data onto the recovered direction and compare the captured variance to the classical baseline. A correct top-eigenvector recovery makes the two ratios match.

In [ ]:
def explained_variance(direction: np.ndarray, X_data: np.ndarray) -> float:
    proj = X_data @ direction
    return float(np.var(proj, ddof=1) / np.trace(X_data.T @ X_data) * X_data.shape[0])


ev_classical = explained_variance(w_classical, X)
ev_quantum = explained_variance(w_quantum, X)
ev_random = explained_variance(np.random.randn(D) / np.sqrt(D), X)

print(f"explained variance, top component (classical PCA): {ev_classical:.4f}")
print(f"explained variance, top component (quantum PCA):   {ev_quantum:.4f}")
print(f"explained variance, random direction (baseline):   {ev_random:.4f}")

### Sample-complexity scaling

Sweeping `M_matrix`, the infidelity $1 - |\langle \vec w_q | \vec w_c \rangle|$ should decrease towards zero with the same $D^2/M$ scaling that drives the matrix-element sketch.

In [ ]:
M_values = np.unique(np.logspace(3, 5.5, 8, dtype=int))
n_repeats = 6
infid = np.empty(M_values.size)
for k, M_k in enumerate(M_values):
    overlaps = []
    for trial in range(n_repeats):
        rng_run = np.random.default_rng(200 + k * 100 + trial)
        w_run, _ = quantum_pca_top_eigenvector(
            X,
            g,
            M_matrix=int(M_k),
            spectral_gap_lower=lambda_second / lambda_top,
            spectral_gap_upper=1.0,
            rng=rng_run,
        )
        overlaps.append(abs(w_run @ w_classical))
    infid[k] = 1.0 - np.mean(overlaps)

c_fit = float(np.median(infid * M_values / (D * D)))

fig, ax = plt.subplots(figsize=(5, 4))
ax.loglog(M_values, infid, "o-", label=r"$1 - |\langle \hat{w}_q | \hat{w}_c \rangle|$")
ax.loglog(
    M_values,
    c_fit * D * D / M_values,
    "--",
    color="C0",
    alpha=0.5,
    label=rf"${c_fit:.2f}\,D^2/M$ guide",
)
ax.set_xlabel("Number of matrix-element samples M")
ax.set_ylabel("Infidelity")
ax.set_title(f"QSVT-projector PCA, $D = {D}$")
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### Classiq circuit fragment

Same matrix-element-sketch fragment as the LS-SVM notebook — the QSVT-projector layer in this notebook only changes the polynomial coefficients, not the underlying block-encoding circuit. We synthesise the matrix-element-sketch portion at $D = 4$.

In [ ]:
D_circuit = 4
NUM_QUBITS_CIRCUIT = int(np.log2(D_circuit))
A_circuit = X[:, :D_circuit].T @ X[:, :D_circuit]
A_circuit_norm = A_circuit / np.max(np.abs(A_circuit))

theta_circuit = 0.05
M_circuit = 60
rng_c = np.random.default_rng(13)
samples_circuit = np.column_stack(
    [
        rng_c.integers(0, D_circuit, size=M_circuit),
        rng_c.integers(0, D_circuit, size=M_circuit),
    ]
)
angles_circuit = (
    theta_circuit
    * D_circuit
    * D_circuit
    / M_circuit
    * A_circuit_norm[samples_circuit[:, 0], samples_circuit[:, 1]]
).tolist()


@qfunc
def main(qvar_i: Output[QNum], qvar_j: Output[QNum]) -> None:
    allocate(NUM_QUBITS_CIRCUIT, qvar_i)
    allocate(NUM_QUBITS_CIRCUIT, qvar_j)
    hadamard_transform(qvar_i)
    hadamard_transform(qvar_j)
    matrix_element_oracle_sketch(
        angles_circuit,
        samples_circuit[:, 0].tolist(),
        samples_circuit[:, 1].tolist(),
        qvar_i,
        qvar_j,
    )


qprog_pca = synthesize(main)
show(qprog_pca)

## Discussion

* **Why a guiding vector?** With no constraint on the input state, the QSVT projector $\Pi_\text{top}$ acts on every basis vector — including the zero overlap with $\vec w$. The guiding vector $\vec g$ ensures the projector has something to amplify; the closer $|\langle \vec g | \vec w\rangle|$ is to 1, the smaller the post-projector renormalisation factor and the easier the readout.

* **Spectral-gap dependence.** The polynomial degree scales as $\widetilde{\mathcal{O}}(1/\Delta)$ in the (normalised) spectral gap $\Delta$. Smaller gaps require steeper transitions and hence higher-degree polynomials — exactly the same structure as the inversion case scales with $\kappa$.

* **Dynamic-data extension.** Theorem 6 of [[1]](#ref1) shows that for a *time-varying* feature distribution refreshing on a $\widetilde{\mathcal{O}}(D)$ timescale, the QOS-PCA pipeline retains its sample efficiency while any classical machine of size $\Omega(D^{0.99})$ requires super-polynomially many samples — the source of the exponential separation.

* **Where the rest lives.** `quantum_pca_top_eigenvector` is promoted to `quantum_ml.py` in the next refactor pass. The [`qos_examples.ipynb`](qos_examples.ipynb) notebook then imports the routine in a single line.

## References

<a id='ref1'></a>
[[1]](#ref1) Zhao, H., Zlokapa, A., Neven, H., Babbush, R., Preskill, J., McClean, J. R., and Huang, H.-Y. *Exponential quantum advantage in processing massive classical data.* arXiv:2604.07639 (2026). https://arxiv.org/abs/2604.07639

<a id='ref2'></a>
[[2]](#ref2) Huang, H.-Y., Kueng, R., and Preskill, J. *Predicting many properties of a quantum system from very few measurements.* Nature Physics 16, 1050–1057 (2020). https://doi.org/10.1038/s41567-020-0932-7 — arXiv: https://arxiv.org/abs/2002.08953

<a id='ref4'></a>
[[4]](#ref4) Gilyén, A., Su, Y., Low, G. H., and Wiebe, N. *Quantum singular value transformation and beyond: exponential improvements for quantum matrix arithmetics.* In Proceedings of the 51st Annual ACM SIGACT Symposium on Theory of Computing, 193–204 (2019). https://doi.org/10.1145/3313276.3316366 — arXiv: https://arxiv.org/abs/1806.01838